In [1]:
!pip install torch_geometric

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^
    torch (>=1.10.0+cu113<1.11.0)
           ~~~~~~~~^


In [2]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import scipy.io
from FVE_GCN_utils import load_surface_mesh
from matplotlib.tri import Triangulation


/opt/conda/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This notebook has LASSO/Random forest data pipeline for brain visualization. Braing mapping and PCA linear regression results are in R scripts.

# LR


In [9]:
wd = Path("/niddk-data-central/mae_hr/FVE")
SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")
output_dir = wd / "LR_output"

B = 50  
top_percents = [0.05, 0.10, 0.15, 0.20]  # Multiple k values to analyze

# Feature dimensions
N_VERTICES_PER_HEMI = 10242  # vertices per hemisphere (0-10241)
N_VERTICES = N_VERTICES_PER_HEMI * 2  # 20484 total
N_COVARIATES = 2  # age and sex
N_FEATURES_REGULAR = N_VERTICES + N_COVARIATES  # 20486
N_FEATURES_PARTIAL = N_VERTICES  # 20484

model_types = [
    'LASSO', 'LASSO_partial', 'LASSO_partial_tsa',
    'Ridge', 'Ridge_partial', 'Ridge_partial_tsa'
]

# Model configurations
MODEL_CONFIGS = {
    'LASSO': {'type': 'regular', 'n_features': N_FEATURES_REGULAR},
    'LASSO_partial': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'LASSO_partial_tsa': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'Ridge': {'type': 'regular', 'n_features': N_FEATURES_REGULAR},
    'Ridge_partial': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL},
    'Ridge_partial_tsa': {'type': 'partial', 'n_features': N_FEATURES_PARTIAL}
}


def load_coefficients_from_csv(model_type, output_dir, B):
    filepath = output_dir / f'coefficients_{model_type}_boot{B}.csv'
    
    df = pd.read_csv(filepath)
    
    coef_dict = {}
    for b in range(1, B + 1):
        col_name = f'b{b}'
        if col_name in df.columns:
            coef_dict[b] = df[col_name].values
    
    return coef_dict


def get_top_features(coefficients, k=0.10):
    abs_coefs = np.abs(coefficients)
    n_features = len(abs_coefs)
    n_top = int(np.ceil(n_features * k))
    top_indices = np.argsort(abs_coefs)[::-1][:n_top]
    
    return top_indices


def extract_feature_importance(output_dir, B, model_configs, k):
    feature_counts = {}
    
    print(f"\nExtracting feature importance for k={k} (top {int(k*100)}%)...")
    
    for model_name, config in model_configs.items():
        n_features = config['n_features']
        feature_counts[model_name] = np.zeros(n_features)
    
        coef_dict = load_coefficients_from_csv(model_name, output_dir, B)
        
        for b in range(1, B + 1):
            if b in coef_dict:
                coefficients = coef_dict[b]
                
                # Get top features (includes age/sex for regular models)
                top_indices = get_top_features(coefficients, k=k)
                
                # Increment count for these features
                feature_counts[model_name][top_indices] += 1
                
        n_selected = np.sum(feature_counts[model_name] > 0)
        print(f"  {model_name}: Total features selected at least once: {n_selected}")
        print(f"  {model_name}: Max selection count: {int(feature_counts[model_name].max())}")
    
    return feature_counts


def create_feature_dataframes(feature_counts, model_configs):
    feature_dfs = {}
    
    for model_name, counts in feature_counts.items():
        n_features = len(counts)
        config = model_configs[model_name]
        
        # Create base dataframe
        df = pd.DataFrame({
            'feature_id': np.arange(n_features),
            'count': counts,
            'proportion': counts / B,
            'model': model_name
        })
        
        # Add feature type and hemisphere labels
        if config['type'] == 'regular':
            # 10242 left + 10242 right + age + sex
            feature_types = ['vertex'] * N_VERTICES + ['age', 'sex']
            hemispheres = ['left'] * N_VERTICES_PER_HEMI + ['right'] * N_VERTICES_PER_HEMI + ['NA', 'NA']
            vertex_nums = list(range(N_VERTICES_PER_HEMI)) * 2 + [-1, -1]
        else:
            # 10242 left + 10242 right vertices only
            feature_types = ['vertex'] * N_VERTICES
            hemispheres = ['left'] * N_VERTICES_PER_HEMI + ['right'] * N_VERTICES_PER_HEMI
            vertex_nums = list(range(N_VERTICES_PER_HEMI)) * 2
        
        df['feature_type'] = feature_types
        df['hemisphere'] = hemispheres
        df['vertex_num'] = vertex_nums
        
        feature_dfs[model_name] = df
    
    return feature_dfs


def save_results(feature_counts, feature_dfs, output_dir, B, k):
    k_str = f"{int(k*100)}pct"
    k_str_plot = f"{int(k*100)}%"
    
    np.save(output_dir / f'vis_output/feature_counts_boot{B}_k{k_str}.npy', feature_counts)
    
    # Save as pickle
    with open(output_dir / f'vis_output/feature_importance_boot{B}_k{k_str}.pkl', 'wb') as f:
        pickle.dump({
            'feature_counts': feature_counts,
            'feature_dfs': feature_dfs,
            'B': B,
            'k': k
        }, f)
    
    print(f"  Saved: feature_counts_boot{B}_k{k_str}.npy")
    print(f"  Saved: feature_importance_boot{B}_k{k_str}.pkl")


def create_summary_statistics(feature_dfs, B, output_dir, k):
    summary_data = []
    
    for model_name, df in feature_dfs.items():
        brain_features = df[df['feature_type'] == 'vertex']
        
        summary_data.append({
            'Model': model_name,
            'k': k,
            'n_total_Vertices_Selected': int(np.sum(brain_features['count'] > 0)),
            'n_vertices>50pct': int(np.sum(brain_features['count'] >= B * 0.5)),
            'n_vertices>75pct': int(np.sum(brain_features['count'] >= B * 0.75)),
            'n_vertices=50times': int(np.sum(brain_features['count'] == B)),
            'max_selection_count': int(brain_features['count'].max())
        })
        
        # For regular models, add covariate info
        if model_name in ['LASSO', 'Ridge']:
            age_count = df[df['feature_type'] == 'age']['count'].values[0]
            sex_count = df[df['feature_type'] == 'sex']['count'].values[0]
            summary_data[-1]['age_selected_count'] = int(age_count)
            summary_data[-1]['sex_selected_count'] = int(sex_count)
    
    summary_df = pd.DataFrame(summary_data)
    
    k_str = f"{int(k*100)}pct"
    k_str_plot = f"{int(k*100)}%"

    summary_df.to_csv(output_dir / f'vis_output/feature_importance_summary_boot{B}_k{k_str}.csv', index=False)
    
    
    return summary_df


def plot_selection_histograms(feature_dfs, output_dir, B, k):
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    k_str = f"{int(k*100)}pct"
    k_str_plot = f"{int(k*100)}%"
    
    for idx, (model_name, df) in enumerate(feature_dfs.items()):
        ax = axes[idx]
        
        # Get brain vertices that were selected at least once
        brain_data = df[df['feature_type'] == 'vertex']
        selected_counts = brain_data[brain_data['count'] > 0]['count'].values
        
        if len(selected_counts) > 0:
            ax.hist(selected_counts, bins=min(50, B), color='steelblue', 
                   alpha=0.7, edgecolor='black')
            ax.axvline(B * 0.5, color='red', linestyle='--', linewidth=2, 
                      label=f'50% ({B*0.5:.0f})')
            ax.axvline(B * 0.75, color='darkred', linestyle='--', linewidth=2, 
                      label=f'75% ({B*0.75:.0f})')
        
        ax.set_xlabel('Number of Times Selected', fontsize=10)
        ax.set_ylabel('Number of Vertices', fontsize=10)
        ax.set_title(f'{model_name}\n{len(selected_counts)} vertices selected', 
                    fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.suptitle(f'Selection Histograms (k={k}, top {k_str})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / f'vis_output/LR_selection_histograms_boot{B}_k{k_str}.png', 
               dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: LR_selection_histograms_boot{B}_k{k_str}.png")


def plot_model_comparison(summary_df, output_dir, B, k):
    fig, ax = plt.subplots(figsize=(14, 6))
    
    k_str = f"{int(k*100)}pct"
    
    models = summary_df['Model'].values
    x = np.arange(len(models))
    width = 0.2
    
    ax.bar(x - 1.5*width, summary_df['n_total_Vertices_Selected'], width, 
           label='Any selection', alpha=0.8, color='lightblue')
    ax.bar(x - 0.5*width, summary_df['n_vertices>50pct'], width, 
           label='>50% iterations', alpha=0.8, color='orange')
    ax.bar(x + 0.5*width, summary_df['n_vertices>75pct'], width, 
           label='>75% iterations', alpha=0.8, color='red')
    ax.bar(x + 1.5*width, summary_df['n_vertices=50times'], width, 
           label='All iterations', alpha=0.8, color='darkred')
    
    ax.set_xlabel('Model Type', fontsize=12)
    ax.set_ylabel('Number of Vertices', fontsize=12)
    ax.set_title(f'Feature Consistency Across Models (B={B}, k={k})', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=0, ha='right')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / f'vis_output/LR_model_comparison_boot{B}_k{k_str}.png', 
               dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: LR_model_comparison_boot{B}_k{k_str}.png")


def plot_brain_heatmap(feature_dfs, model_types, B, output_dir, k):
    
    fig, axes = plt.subplots(6, 1, figsize=(16, 24))
    
    k_str = f"{int(k*100)}pct"
    k_str_plot = f"{int(k*100)}%"

    # Create white to red colormap
    cmap = LinearSegmentedColormap.from_list('white_red', ['white', 'red'])
    
    # Display names for models
    model_display_names = {
        'LASSO': 'LASSO',
        'LASSO_partial': 'LASSO partial',
        'LASSO_partial_tsa': 'LASSO TSA partial',
        'Ridge': 'Ridge',
        'Ridge_partial': 'Ridge partial',
        'Ridge_partial_tsa': 'Ridge TSA partial'
    }
    
    for idx, model_type in enumerate(model_types):
        if model_type not in feature_dfs:
            continue
        
        # Extract hemisphere data
        df = feature_dfs[model_type]
        brain_features = df[df['feature_type'] == 'vertex']
        
        lh_data = brain_features[brain_features['hemisphere'] == 'left'].copy()
        rh_data = brain_features[brain_features['hemisphere'] == 'right'].copy()
        
        lh_counts = np.zeros(N_VERTICES_PER_HEMI)
        rh_counts = np.zeros(N_VERTICES_PER_HEMI)
        
        for _, row in lh_data.iterrows():
            lh_counts[int(row['vertex_num'])] = row['count']
        
        for _, row in rh_data.iterrows():
            rh_counts[int(row['vertex_num'])] = row['count']
        
        # Plot in subplot
        ax = axes[idx]
        
        # Stack left and right hemispheres horizontally
        combined_counts = np.hstack([lh_counts.reshape(1, -1), rh_counts.reshape(1, -1)])
        
        im = ax.imshow(combined_counts, aspect='auto', cmap=cmap, vmin=0, vmax=B)
        
        ax.axvline(x=N_VERTICES_PER_HEMI - 0.5, color='black', linewidth=2, linestyle='--')
        
        # Labels - use display name
        display_name = model_display_names.get(model_type, model_type)
        ax.set_ylabel(f'{display_name}', fontsize=14, fontweight='bold')
        
        # Only show x-axis label on bottom plot
        if idx == len(model_types) - 1:
            ax.set_xlabel('Vertex Index', fontsize=11)
            ax.text(N_VERTICES_PER_HEMI/2, -0.15, 'Left\nHemisphere', 
                    transform=ax.get_xaxis_transform(), fontsize=12, 
                    ha='center', va='center')
            ax.text(N_VERTICES_PER_HEMI + N_VERTICES_PER_HEMI/2, -0.15, 'Right\nHemisphere', 
                    transform=ax.get_xaxis_transform(), fontsize=12, 
                    ha='center', va='center')
        else:
            ax.set_xticklabels([])
    
    # Add single colorbar for all subplots
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax, label=f'Selection Count (out of {B})')
    
    plt.suptitle(f'LASSO and Ridge top {k_str_plot} vertices selection', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.savefig(output_dir / f'vis_output/LR_brain_heatmap_boot{B}_k{k_str}.png', dpi=450, bbox_inches='tight')
    plt.close()
    

In [10]:
print("="*70)
print("FEATURE IMPORTANCE ANALYSIS - MULTIPLE K VALUES")
print("="*70)

# Check for coefficient files
coef_files = list(output_dir.glob('coefficients_*_boot_50.csv'))
print(f"\nFound {len(coef_files)} coefficient files")

# Store all summaries
all_summaries = []

# Loop through each k value
for k in top_percents:
    print(f"\n{'='*70}")
    print(f"RUNNING ANALYSIS FOR k = {k} (top {int(k*100)}%)")
    print(f"{'='*70}")
    
    # Extract feature importance
    feature_counts = extract_feature_importance(output_dir, B, MODEL_CONFIGS, k)
    
    # Create feature dataframes
    feature_dfs = create_feature_dataframes(feature_counts, MODEL_CONFIGS)
    
    # Create summary statistics
    summary_df = create_summary_statistics(feature_dfs, B, output_dir, k)
    all_summaries.append(summary_df)
    
    # Save results
    save_results(feature_counts, feature_dfs, output_dir, B, k)
    
    # Visualizations
    print(f"\nGenerating visualizations for k={k}...")
    plot_selection_histograms(feature_dfs, output_dir, B, k)
    plot_model_comparison(summary_df, output_dir, B, k)
    plot_brain_heatmap(feature_dfs, model_types, B, output_dir, k)
    
    print(f"\nCompleted analysis for k={k}")

# Create combined summary across all k values
print(f"\n{'='*70}")
print("CREATING COMBINED SUMMARY")
print(f"{'='*70}")

combined_summary = pd.concat(all_summaries, ignore_index=True)
combined_summary.to_csv(output_dir / f'vis_output/feature_importance_summary_boot{B}_all_k.csv', index=False)
print(f"Saved: feature_importance_summary_boot{B}_all_k.csv")

print(f"\n{'='*70}")
print("ANALYSIS COMPLETE")
print(f"{'='*70}")
print(f"\nAnalyzed {len(top_percents)} different k values: {top_percents}")
print(f"Total output files generated: {len(top_percents) * 6 + 1}")
print(f"\nAll files saved to: {output_dir / 'vis_output'}")

FEATURE IMPORTANCE ANALYSIS - MULTIPLE K VALUES

Found 0 coefficient files

RUNNING ANALYSIS FOR k = 0.05 (top 5%)

Extracting feature importance for k=0.05 (top 5%)...
  LASSO: Total features selected at least once: 11272
  LASSO: Max selection count: 50
  LASSO_partial: Total features selected at least once: 11668
  LASSO_partial: Max selection count: 44
  LASSO_partial_tsa: Total features selected at least once: 12926
  LASSO_partial_tsa: Max selection count: 45
  Ridge: Total features selected at least once: 5504
  Ridge: Max selection count: 50
  Ridge_partial: Total features selected at least once: 5680
  Ridge_partial: Max selection count: 50
  Ridge_partial_tsa: Total features selected at least once: 6272
  Ridge_partial_tsa: Max selection count: 50
  Saved: feature_counts_boot50_k5pct.npy
  Saved: feature_importance_boot50_k5pct.pkl

Generating visualizations for k=0.05...
  Saved: LR_selection_histograms_boot50_k5pct.png
  Saved: LR_model_comparison_boot50_k5pct.png


/tmp/ipykernel_9634/2117342823.py:316: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])



Completed analysis for k=0.05

RUNNING ANALYSIS FOR k = 0.1 (top 10%)

Extracting feature importance for k=0.1 (top 10%)...
  LASSO: Total features selected at least once: 12015
  LASSO: Max selection count: 50
  LASSO_partial: Total features selected at least once: 12390
  LASSO_partial: Max selection count: 50
  LASSO_partial_tsa: Total features selected at least once: 15729
  LASSO_partial_tsa: Max selection count: 47
  Ridge: Total features selected at least once: 9455
  Ridge: Max selection count: 50
  Ridge_partial: Total features selected at least once: 9596
  Ridge_partial: Max selection count: 50
  Ridge_partial_tsa: Total features selected at least once: 10366
  Ridge_partial_tsa: Max selection count: 50
  Saved: feature_counts_boot50_k10pct.npy
  Saved: feature_importance_boot50_k10pct.pkl

Generating visualizations for k=0.1...
  Saved: LR_selection_histograms_boot50_k10pct.png
  Saved: LR_model_comparison_boot50_k10pct.png

Completed analysis for k=0.1

RUNNING ANALYSIS F

## Export

In [11]:
all_exported = []

for k in top_percents:
    k_str = f"{int(k*100)}pct"
    pkl_file = output_dir / f'vis_output/feature_importance_boot{B}_k{k_str}.pkl'
    
    print(f"\n{'='*70}")
    print(f"Processing k = {k} (top {k_str})")
    print(f"{'='*70}")
    
    # Check if pickle file exists
    if not pkl_file.exists():
        print(f"WARNING: File not found: {pkl_file}")
        print(f"Skipping k={k}")
        continue
    
    # Load pickle file
    print(f"Loading: {pkl_file.name}")
    with open(pkl_file, 'rb') as f:
        data = pickle.load(f)
        feature_dfs = data['feature_dfs']
        loaded_k = data.get('k', 'unknown')
    
    print(f"Found {len(feature_dfs)} models for k={loaded_k}")
    
    # Export each model to CSV
    model_count = 0
    for model_name, df in feature_dfs.items():
        output_file = output_dir / f'vis_output/feature_importance_{model_name}_boot{B}_k{k_str}.csv'
        df.to_csv(output_file, index=False)
        print(f"  Saved: feature_importance_{model_name}_boot{B}_k{k_str}.csv")
        all_exported.append({
            'k': k,
            'k_str': k_str,
            'model': model_name,
            'filename': output_file.name,
            'n_features': len(df)
        })
        model_count += 1
    
    print(f"Exported {model_count} models for k={k}")


Processing k = 0.05 (top 5pct)
Loading: feature_importance_boot50_k5pct.pkl
Found 6 models for k=0.05
  Saved: feature_importance_LASSO_boot50_k5pct.csv
  Saved: feature_importance_LASSO_partial_boot50_k5pct.csv
  Saved: feature_importance_LASSO_partial_tsa_boot50_k5pct.csv
  Saved: feature_importance_Ridge_boot50_k5pct.csv
  Saved: feature_importance_Ridge_partial_boot50_k5pct.csv
  Saved: feature_importance_Ridge_partial_tsa_boot50_k5pct.csv
Exported 6 models for k=0.05

Processing k = 0.1 (top 10pct)
Loading: feature_importance_boot50_k10pct.pkl
Found 6 models for k=0.1
  Saved: feature_importance_LASSO_boot50_k10pct.csv
  Saved: feature_importance_LASSO_partial_boot50_k10pct.csv
  Saved: feature_importance_LASSO_partial_tsa_boot50_k10pct.csv
  Saved: feature_importance_Ridge_boot50_k10pct.csv
  Saved: feature_importance_Ridge_partial_boot50_k10pct.csv
  Saved: feature_importance_Ridge_partial_tsa_boot50_k10pct.csv
Exported 6 models for k=0.1

Processing k = 0.15 (top 15pct)
Loadin

# RF


In [16]:
rf_partial_importance_1 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_10_1997_partial_feature_importances.csv")
rf_partial_importance_2 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_11_1994_partial_feature_importances.csv")
rf_partial_importance_3 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_30_1013_partial_feature_importances.csv")
rf_partial_importance = pd.concat([rf_partial_importance_1, rf_partial_importance_2, rf_partial_importance_3], ignore_index=True)
rf_partial_tsa_importance = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_50_1013_partial_tsa_feature_importances.csv")
rf_importance = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_50_1013_feature_importances.csv")

In [17]:
rf_partial_r2_1 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_10_1997_partial_R2.csv")
rf_partial_r2_2 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_11_1994_partial_R2.csv")
rf_partial_r2_3 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_30_1013_partial_R2.csv")
rf_partial_r2 = pd.concat([rf_partial_r2_1, rf_partial_r2_2, rf_partial_r2_3], ignore_index=True)
rf_partial_tsa_r2 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_50_1013_partial_tsa_R2.csv")
rf_r2 = pd.read_csv("/niddk-data-central/mae_hr/FVE/rf_output/test_50_1013_R2.csv")

In [19]:
rf_partial_r2 = pd.concat([rf_partial_r2_1, rf_partial_r2_2, rf_partial_r2_3], ignore_index=True)
print(rf_partial_r2.mean(), np.std(rf_partial_r2))

Unnamed: 0    10.12000
0              0.09202
dtype: float64 Unnamed: 0    8.078713
0             0.007924
dtype: float64


In [21]:
rf_partial_importance.to_csv("/niddk-data-central/mae_hr/FVE/rf_output/rf_partial_importance.csv")
rf_partial_tsa_importance.to_csv("/niddk-data-central/mae_hr/FVE/rf_output/rf_partial_tsa_importance.csv")
rf_importance.to_csv("/niddk-data-central/mae_hr/FVE/rf_output/rf_importance.csv")